In [1]:
import os
import warnings
warnings.filterwarnings('ignore', message = ".*Glyph.*")
import gc, json, random, shutil, subprocess, sys, time
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import font_manager
from PIL import Image

QUICK = True

BASE = Path.cwd()
ASSETS = BASE / 'assets'
DATA = BASE / 'data' / 'knowledge'
WORK = BASE / 'work'
for d in ('models', 'chroma', 'images'):
    (WORK / d).mkdir(parents = True, exist_ok = True)

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

IMG_TYPE = torch.float16 if DEVICE in ('cuda', 'mps') else torch.float32

installed = {f.name for f in font_manager.fontManager.ttflist}
for name in ('AppleGothic', 'Malgun Gothic', 'NanumGothic'):
    if name in installed:
        plt.rcParams['font.family'] = name
        break

plt.rcParams['axes.unicode_minus'] = False


In [2]:
def set_seed(seed = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def free_memory():
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

def show_images(images, titles = None, cols = None, size = 3.2):
    cols = cols or len(images)
    rows = (len(images) + cols -1) // cols
    fig, axes = plt.subplots(rows, cols, figsize = (size * cols, size * rows), squeeze = False)
    for ax in axes.ravel():
        ax.axis('off')

    for i, im in enumerate(images):
        axes.ravel()[i].imshow(im)
        if titles:
            axes.ravel()[i].set_title(titles[i], fontsize = 10)

    plt.tick_params()
    plt.show()

set_seed(42)
print(DEVICE, QUICK)
print(BASE)

cuda True
c:\Users\KDS-22\Documents\17_Study\17_rag\DAY1_261002


In [4]:
from transformers import AutoTokenizer, AutoModel

EMB_NAME = 'intfloat/multilingual-e5-small'
tok = AutoTokenizer.from_pretrained(EMB_NAME)
emb_model = AutoModel.from_pretrained(EMB_NAME).eval().to(DEVICE)

sentences = ['나는 강아지를 정말 좋아합니다.', '금요일은 행복한 날이에요']
enc = tok(sentences, padding = True, truncation = True, return_tensors = 'pt')

print(enc['input_ids'])
for i in range(len(sentences)):
    print(tok.convert_ids_to_tokens(enc['input_ids'][i]))
print(enc['attention_mask'])



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tensor([[     0,  37231,  13564,   3222,  49067,  34682,  67738,   8642,      5,
              2],
        [     0,  36527,   5144,   1599,    697, 169384,  21858, 190278,      2,
              1]])
['<s>', '▁나는', '▁강', '아', '지를', '▁정말', '▁좋아', '합니다', '.', '</s>']
['<s>', '▁금', '요', '일', '은', '▁행복한', '▁날', '이에요', '</s>', '<pad>']
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 0]])


In [5]:
with torch.no_grad():
    out = emb_model(**enc.to(DEVICE))

emb = out.last_hidden_state

print(tuple(emb.shape))
print(emb[0, 0, :6].cpu().numpy().round(3))

(2, 10, 384)
[ 0.166 -0.037 -0.112 -0.297  0.428 -0.065]


In [ ]:
import torch.nn.functional as F

def embed(texts, prefix = 'query', batch_size = 32):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = [prefix + t for t in texts[i:i + batch_size]]
        enc = tok(batch, padding = True, truncation=True, max_length=512, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            hidden = emb_model(**enc).last_hidden_state
        mask = enc['attention_mask'].unsqueeze(-1).float()
        pooled = (hidden * mask).sum(1) / mask.sum(1)
        vectors.append(F.normalize(pooled, p = 2, dim = 1).cpu())
    return torch.cat(vectors)

test = ['나는 강아지를 정말 좋아해요.',
        '우리 집 멍멍이가 세상에서 제일 귀여워요.',
        '주식 시장이 크게 폭락했습니다.']

vecs = embed(test)
sim = vecs @vecs.T

fig, ax = plt.subplots(figsize = (5.2, 4))
im = ax.imshow(sim.numpy(), cmap = 'Blues', vmin = 0.6, vmax = 1)
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(['강아지좋아함', '멍멍이귀여움', '주식폭락'])
ax.set_yticklabels(['강아지좋아함', '멍멍이귀여움', '주식폭락'])

for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha = 'center', va = 'center', color = 'white' if sim[i, j] > 0.85 else 'black')

ax.set_title('문장 사이의 뜻이 가까울 수록 1과 비슷')

plt.colorbar(im)
plt.tight_layout()
plt.show()
